# GPT-2 nano on Colab T4

Trains a ~10.6M parameter GPT-2 style transformer on TinyShakespeare.

**Estimated wall time on a Colab T4 (16 GB):**

| iters | batch | block | approx time | final val loss |
|------:|------:|------:|------------:|---------------:|
| 2,000 | 64    | 256   | ~6–8 min    | ~1.65          |
| 5,000 | 64    | 256   | ~15–22 min  | ~1.50          |
| 10,000| 64    | 256   | ~30–45 min  | ~1.45          |

Per-iter time on T4 with fp16 + flash SDPA is ~150–250 ms. Numbers vary with the specific T4 SKU Colab gives you.

**Before running:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU`.

## 1. Verify the GPU

In [ ]:
!nvidia-smi

## 2. Get the project

Pick **one** of the two cells below.

**Option A — clone from GitHub** (replace with your repo URL once pushed):

In [ ]:
# !git clone https://github.com/<you>/gpt-2-nano.git
# %cd gpt-2-nano

**Option B — upload the project files manually** to Colab's file browser and `cd` into the folder:

In [ ]:
import os
if not os.path.exists('model.py'):
    raise RuntimeError('Run this notebook from the gpt-2-nano project root, or clone the repo first.')
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.')))

## 3. Install dependencies

Colab already ships with a CUDA-built PyTorch, so we only need the small extras.

In [ ]:
!pip -q install tiktoken requests

## 4. Prepare the dataset

Downloads TinyShakespeare (~1 MB) and tokenizes it with the GPT-2 BPE tokenizer.

In [ ]:
from data import prepare_tinyshakespeare
prepare_tinyshakespeare()

## 5. Sanity-check the model

One forward pass on random input — confirms the architecture is wired up and tells you the parameter count.

In [ ]:
import torch
from config import ModelConfig
from model import GPT

mcfg = ModelConfig()
model = GPT(mcfg).cuda()
print(f'parameters: {model.num_parameters() / 1e6:.2f}M')
x = torch.randint(0, mcfg.vocab_size, (2, 32), device='cuda')
logits, loss = model(x, x)
print('logits shape:', logits.shape, 'loss:', loss.item())
del model; torch.cuda.empty_cache()

## 6. Train

5,000 iters at the default config takes ~15–22 min on a T4. Lower `--max_iters` for a faster smoke test.

In [ ]:
!python train.py --max_iters 5000 --batch_size 64

## 7. Sample from the trained model

In [ ]:
!python sample.py --prompt "ROMEO:" --max_new_tokens 300 --temperature 0.8 --top_k 200

## 8. (Optional) Save the checkpoint to Google Drive

Colab disks are wiped on disconnect — copy `out/ckpt.pt` to Drive if you want to keep it.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/gpt-2-nano && cp out/ckpt.pt /content/drive/MyDrive/gpt-2-nano/